In [1]:
%pip install -q transformers==4.51.3 datasets==3.5.0 peft==0.15.2 accelerate==1.6.0 trl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.12.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch
import transformers
import datasets
import peft
import accelerate
import trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("TRL:", trl.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
Transformers: 4.51.3
Datasets: 3.5.0
PEFT: 0.15.2
Accelerate: 1.6.0
TRL: 0.19.1
CUDA available: True
GPU: Tesla T4


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os

SFT_ADAPTER_DIR = "/content/drive/MyDrive/ReelRhythm/sft_adapter"

print("SFT adapter exists:", os.path.exists(SFT_ADAPTER_DIR))

if os.path.exists(SFT_ADAPTER_DIR):
    print("\nAdapter files:")
    for filename in os.listdir(SFT_ADAPTER_DIR):
        print(" -", filename)

SFT adapter exists: True

Adapter files:
 - README.md
 - adapter_model.safetensors
 - adapter_config.json
 - vocab.json
 - tokenizer.json
 - merges.txt
 - special_tokens_map.json
 - tokenizer_config.json
 - added_tokens.json


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the original Qwen model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded!")
print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", sum(p.numel() for p in model.parameters()))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded!
GPU: Tesla T4
Parameters: 494032768


In [6]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    SFT_ADAPTER_DIR
)

model.print_trainable_parameters()

print("SFT adapter loaded successfully!")

trainable params: 0 || all params: 502,830,976 || trainable%: 0.0000
SFT adapter loaded successfully!


In [7]:
model.train()

for name, param in model.named_parameters():
    if "lora_" in name:
        param.requires_grad = True

model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [122]:
from google.colab import files

uploaded = files.upload()

Saving combined_test.jsonl to combined_test.jsonl


In [11]:
import json
import pandas as pd

# Load training examples
with open("train_all.jsonl", "r", encoding="utf-8") as file:
    examples = [json.loads(line) for line in file]

# Load reference databases
movies_df = pd.read_csv("movies_clean.csv")
spotify_df = pd.read_csv("spotify_clean.csv")

print("Training examples:", len(examples))
print("Movies:", len(movies_df))
print("Spotify tracks:", len(spotify_df))

print("\nMovie columns:")
print(movies_df.columns.tolist())

print("\nSpotify columns:")
print(spotify_df.columns.tolist())

print("\nExample training prompt:")
print(examples[0]["messages"][-2]["content"])

print("\nExample expected answer:")
print(examples[0]["messages"][-1]["content"])

Training examples: 2320
Movies: 340356
Spotify tracks: 8579

Movie columns:
['id', 'title', 'year', 'genres', 'rating', 'votes']

Spotify columns:
['id', 'title', 'artist', 'popularity', 'artist_popularity', 'artist_followers', 'genres', 'album', 'release_date', 'duration_min', 'explicit']

Example training prompt:
Recommend 3 Sci-Fi movies and 3 nu metal tracks.

Example expected answer:
Movies:
1. G.O.R.A. (2004) | Genres: Adventure,Comedy,Sci-Fi | Rating: 8.0 | Votes: 72897 | IMDb ID: tt0384116
2. Ready Player One (2018) | Genres: Action,Adventure,Sci-Fi | Rating: 7.4 | Votes: 555427 | IMDb ID: tt1677720
3. Indru Netru Naalai (2015) | Genres: Comedy,Sci-Fi,Thriller | Rating: 8.0 | Votes: 5702 | IMDb ID: tt4806232

Music:
1. Talking to Myself - Linkin Park | Genres: nu metal, rap metal, rock, alternative metal | Popularity: 60 | Year: 2017 | Duration: 3.85 min | Explicit: No | Spotify ID: 7nAfXgeHfDO50upcOjJOaq
2. Posthuman - Marilyn Manson | Genres: industrial metal, industrial rock

In [12]:
# Collect unique training prompts
unique_prompts = set()

for example in examples:
    prompt = example["messages"][-2]["content"]
    unique_prompts.add(prompt)

print("Total unique prompts:", len(unique_prompts))

print("\nSample prompts:\n")

for prompt in list(unique_prompts)[:25]:
    print("-", prompt)

Total unique prompts: 1440

Sample prompts:

- Recommend 5 Musical movies and 3 indie folk tracks.
- Recommend 5 rap tracks with popularity of at least 80.
- Recommend 3 Adventure movies and 3 anime tracks.
- Recommend 3 Documentary movies and 5 contemporary r&b tracks.
- What are 3 songs from the 1990s that you recommend?
- Recommend 5 drama movies from the 1990s rated at least 7.0.
- Recommend 5 indie tracks.
- Recommend 3 Western movies and 3 anime tracks.
- Give me 5 tracks released during the 2020s.
- Give me 3 good country songs.
- Recommend 5 Film-Noir movies and 3 egyptian pop tracks.
- Find 10 clean country songs.
- Give me 5 edm tracks that have a popularity score of 60 or higher.
- Can you recommend 5 animation movies?
- What are 5 movies from the 2020s that you recommend?
- Find 5 clean alternative r&b songs.
- Recommend 5 thriller movies from the 1990s rated at least 7.5.
- Find me 5 comedy movies rated 7.5+.
- Recommend 10 hip hop songs from the 2000s.
- Recommend 5 Famil

In [13]:
from collections import Counter

categories = Counter()
constraints = Counter()
examples_by_constraint = {}

for example in examples:
    prompt = example["messages"][-2]["content"]
    p = prompt.lower()

    # Identify request type
    has_movie = "movie" in p or "film" in p
    has_music = any(word in p for word in ["track", "song", "music"])

    if has_movie and has_music:
        categories["Combined"] += 1
    elif has_movie:
        categories["Movies"] += 1
    elif has_music:
        categories["Music"] += 1

    # Identify constraints
    checks = {
        "Genre": any(word in p for word in ["genre", "sci-fi", "hip hop", "country", "metal", "comedy"]),
        "Year / Decade": any(word in p for word in ["1990s", "2000s", "2010s", "2020s", "released"]),
        "Rating": "rating" in p or "rated" in p or "imdb" in p,
        "Popularity": "popularity" in p or "popular" in p,
        "Explicit content": "clean" in p or "explicit" in p,
        "Duration": "minutes" in p or "long" in p,
        "Quantity": any(str(n) in p for n in [3, 5, 10])
    }

    for name, found in checks.items():
        if found:
            constraints[name] += 1
            examples_by_constraint.setdefault(name, prompt)

print("REQUEST TYPES")
for name, count in categories.items():
    print(f"{name}: {count}")

print("\nCONSTRAINTS")
for name, count in constraints.items():
    print(f"{name}: {count}")

print("\nEXAMPLE OF EACH CONSTRAINT")
for name, prompt in examples_by_constraint.items():
    print(f"\n{name}: {prompt}")

REQUEST TYPES
Combined: 320
Music: 800
Movies: 1200

CONSTRAINTS
Genre: 251
Quantity: 2271
Duration: 106
Year / Decade: 576
Rating: 511
Explicit content: 211
Popularity: 211

EXAMPLE OF EACH CONSTRAINT

Genre: Recommend 3 Sci-Fi movies and 3 nu metal tracks.

Quantity: Recommend 3 Sci-Fi movies and 3 nu metal tracks.

Duration: Find 5 tracks that are under 2.5 minutes long.

Year / Decade: I want country music from the 2010s. Give me 5 recommendations.

Rating: Find 3 comedy movies from the 1980s with an IMDb rating of 7.5+.

Explicit content: Recommend 5 non-explicit songs.

Popularity: Recommend 10 tracks by artists with popularity of at least 50.


In [14]:
# Make sure IDs are strings
movies_df["id"] = movies_df["id"].astype(str)
spotify_df["id"] = spotify_df["id"].astype(str)

# Create movie lookup
movie_lookup = {}

for row in movies_df.itertuples(index=False):
    movie_lookup[row.id] = {
        "title": row.title,
        "year": row.year,
        "genres": str(row.genres).lower(),
        "rating": row.rating
    }

# Create Spotify lookup
spotify_lookup = {}

for row in spotify_df.itertuples(index=False):
    spotify_lookup[row.id] = {
        "title": row.title,
        "artist": row.artist,
        "genres": str(row.genres).lower(),
        "popularity": row.popularity,
        "year": row.release_date,
        "duration": row.duration_min,
        "explicit": row.explicit
    }

print("Movie lookup entries:", len(movie_lookup))
print("Spotify lookup entries:", len(spotify_lookup))

# Test a movie
print("\nExample movie:")
print(next(iter(movie_lookup.items())))

# Test a Spotify track
print("\nExample track:")
print(next(iter(spotify_lookup.items())))

Movie lookup entries: 340356
Spotify lookup entries: 8579

Example movie:
('tt0000009', {'title': 'Miss Jerry', 'year': 1894, 'genres': 'romance', 'rating': 5.3})

Example track:
('3EJS5LyekDim1Tf5rBFmZl', {'title': 'Trippy Mane (ft. Project Pat)', 'artist': 'Diplo', 'genres': 'moombahton', 'popularity': 0, 'year': '2025-10-31', 'duration': 1.55, 'explicit': True})


In [15]:
import re

def extract_movie_ids(text):
    return re.findall(r"IMDb ID:\s*(tt\d+)", text)

def extract_spotify_ids(text):
    return re.findall(r"Spotify ID:\s*([A-Za-z0-9]+)", text)


# Test using an existing training answer
test_answer = examples[0]["messages"][-1]["content"]

movie_ids = extract_movie_ids(test_answer)
spotify_ids = extract_spotify_ids(test_answer)

print("Extracted movie IDs:")
print(movie_ids)

print("\nExtracted Spotify IDs:")
print(spotify_ids)

print("\nMovie IDs found in database:")
print(sum(1 for movie_id in movie_ids if movie_id in movie_lookup))

print("\nSpotify IDs found in database:")
print(sum(1 for track_id in spotify_ids if track_id in spotify_lookup))

Extracted movie IDs:
['tt0384116', 'tt1677720', 'tt4806232']

Extracted Spotify IDs:
['7nAfXgeHfDO50upcOjJOaq', '1mI12kERORxUPtlrQe0OMb', '7AB0cUXnzuSlAnyHOqmrZr']

Movie IDs found in database:
3

Spotify IDs found in database:
3


In [16]:
import re

def get_requested_counts(prompt):

    prompt_lower = prompt.lower()

    movie_count = None
    music_count = None

    # Find quantities associated with movies or films
    movie_match = re.search(
        r"\b(\d+)\s+(?:[\w&+-]+\s+){0,3}(?:movies|films)\b",
        prompt_lower
    )

    # Find quantities associated with tracks or songs
    music_match = re.search(
        r"\b(\d+)\s+(?:[\w&+-]+\s+){0,3}(?:tracks|songs)\b",
        prompt_lower
    )

    if movie_match:
        movie_count = int(movie_match.group(1))

    if music_match:
        music_count = int(music_match.group(1))

    # Handle prompts that say "Give me 5 recommendations"
    generic_match = re.search(
        r"\b(\d+)\s+recommendations\b",
        prompt_lower
    )

    if generic_match:
        if "movie" in prompt_lower or "film" in prompt_lower:
            movie_count = int(generic_match.group(1))
        elif any(word in prompt_lower for word in ["music", "song", "track"]):
            music_count = int(generic_match.group(1))

    return movie_count, music_count


# Test a few examples
test_prompts = [
    "Recommend 3 Sci-Fi movies and 3 nu metal tracks.",
    "Recommend 5 drama movies from the 1990s.",
    "Find 10 clean country songs.",
    "I want country music from the 2010s. Give me 5 recommendations."
]

for prompt in test_prompts:
    movie_count, music_count = get_requested_counts(prompt)

    print(prompt)
    print("Movies:", movie_count, "| Music:", music_count)
    print()

Recommend 3 Sci-Fi movies and 3 nu metal tracks.
Movies: 3 | Music: 3

Recommend 5 drama movies from the 1990s.
Movies: 5 | Music: None

Find 10 clean country songs.
Movies: None | Music: 10

I want country music from the 2010s. Give me 5 recommendations.
Movies: None | Music: 5



In [18]:
def get_requested_decade(prompt):

    match = re.search(r"\b(19\d0s|20\d0s)\b", prompt.lower())

    if match:
        start_year = int(match.group(1)[:-1])
        end_year = start_year + 9

        return start_year, end_year

    return None, None


# Test examples
test_prompts = [
    "Recommend 5 drama movies from the 1990s.",
    "Give me 10 hip hop songs from the 2000s.",
    "What are 5 movies from the 2020s that you recommend?",
    "Recommend 3 comedy movies."
]

for prompt in test_prompts:
    start, end = get_requested_decade(prompt)

    print(prompt)
    print("Decade:", start, "-", end)
    print()

Recommend 5 drama movies from the 1990s.
Decade: 1990 - 1999

Give me 10 hip hop songs from the 2000s.
Decade: 2000 - 2009

What are 5 movies from the 2020s that you recommend?
Decade: 2020 - 2029

Recommend 3 comedy movies.
Decade: None - None



In [21]:
# Collect all available genres from both databases

movie_genres = set()
music_genres = set()

for genres in movies_df["genres"].dropna():
    for genre in str(genres).split(","):
        movie_genres.add(genre.strip().lower())

for genres in spotify_df["genres"].dropna():
    for genre in str(genres).split(","):
        music_genres.add(genre.strip().lower())


def get_requested_genres(prompt):

    prompt_lower = prompt.lower()

    requested_movie_genres = []
    requested_music_genres = []

    # Split the prompt into separate requests
    parts = re.split(r"\band\b", prompt_lower)

    for part in parts:

        has_movie_request = (
            "movie" in part or "film" in part
        )

        has_music_request = any(
            word in part
            for word in ["track", "song", "music"]
        )

        # Find movie genres in movie requests
        if has_movie_request:
            for genre in movie_genres:
                pattern = r"(?<!\w)" + re.escape(genre) + r"(?!\w)"

                if re.search(pattern, part):
                    requested_movie_genres.append(genre)

        # Find music genres in music requests
        if has_music_request:
            for genre in music_genres:
                pattern = r"(?<!\w)" + re.escape(genre) + r"(?!\w)"

                if re.search(pattern, part):
                    requested_music_genres.append(genre)

    # Remove shorter overlapping genre names
    requested_movie_genres = [
        genre for genre in requested_movie_genres
        if not any(
            genre != other and genre in other
            for other in requested_movie_genres
        )
    ]

    requested_music_genres = [
        genre for genre in requested_music_genres
        if not any(
            genre != other and genre in other
            for other in requested_music_genres
        )
    ]

    return requested_movie_genres, requested_music_genres


# Run the tests again
for prompt in test_prompts:
    movie_genres_found, music_genres_found = get_requested_genres(prompt)

    print(prompt)
    print("Movie genres:", movie_genres_found)
    print("Music genres:", music_genres_found)
    print()

Recommend 3 Sci-Fi movies and 3 nu metal tracks.
Movie genres: ['sci-fi']
Music genres: ['nu metal']

Recommend 5 drama movies from the 1990s.
Movie genres: ['drama']
Music genres: []

Find 10 clean country songs.
Movie genres: []
Music genres: ['country']

Recommend 3 comedy movies and 5 hip hop tracks.
Movie genres: ['comedy']
Music genres: ['hip hop']



In [22]:
def get_requested_rating(prompt):

    prompt_lower = prompt.lower()

    match = re.search(
        r"(?:rated|rating(?: of)?|imdb rating(?: of)?)\s*(?:at least\s*)?(\d+(?:\.\d+)?)\s*(?:or higher|\+)?",
        prompt_lower
    )

    if match:
        return float(match.group(1))

    return None


# Test examples
test_prompts = [
    "Recommend 5 drama movies rated at least 7.0.",
    "What are 10 movies rated 8.5 or higher?",
    "Find 5 comedy movies rated 7.5+.",
    "Recommend 3 Sci-Fi movies."
]

for prompt in test_prompts:
    rating = get_requested_rating(prompt)

    print(prompt)
    print("Minimum rating:", rating)
    print()

Recommend 5 drama movies rated at least 7.0.
Minimum rating: 7.0

What are 10 movies rated 8.5 or higher?
Minimum rating: 8.5

Find 5 comedy movies rated 7.5+.
Minimum rating: 7.5

Recommend 3 Sci-Fi movies.
Minimum rating: None



In [23]:
def get_requested_popularity(prompt):

    prompt_lower = prompt.lower()

    match = re.search(
        r"popularity(?:\s+score)?\s*(?:of\s*)?(?:at least\s*)?(\d+)\s*(?:or higher|\+)?",
        prompt_lower
    )

    if match:
        return int(match.group(1))

    return None


# Test examples
test_prompts = [
    "Recommend 5 rap tracks with popularity of at least 80.",
    "Find 5 popular synthpop songs with Spotify popularity 70+.",
    "Give me 5 edm tracks that have a popularity score of 60 or higher.",
    "Recommend 3 country songs."
]

for prompt in test_prompts:
    popularity = get_requested_popularity(prompt)

    print(prompt)
    print("Minimum popularity:", popularity)
    print()

Recommend 5 rap tracks with popularity of at least 80.
Minimum popularity: 80

Find 5 popular synthpop songs with Spotify popularity 70+.
Minimum popularity: 70

Give me 5 edm tracks that have a popularity score of 60 or higher.
Minimum popularity: 60

Recommend 3 country songs.
Minimum popularity: None



In [24]:
def get_requested_explicit(prompt):

    prompt_lower = prompt.lower()

    if re.search(r"\bclean\b", prompt_lower):
        return False

    if re.search(r"\bexplicit\b", prompt_lower):
        return True

    return None


# Test examples
test_prompts = [
    "Find 10 clean country songs.",
    "Recommend 5 explicit rap tracks.",
    "Give me 3 country songs.",
    "Find 5 clean alternative r&b songs."
]

for prompt in test_prompts:
    explicit = get_requested_explicit(prompt)

    print(prompt)
    print("Explicit requirement:", explicit)
    print()

Find 10 clean country songs.
Explicit requirement: False

Recommend 5 explicit rap tracks.
Explicit requirement: True

Give me 3 country songs.
Explicit requirement: None

Find 5 clean alternative r&b songs.
Explicit requirement: False



In [190]:
def get_requested_duration(prompt):

    prompt_lower = prompt.lower()

    match = re.search(
        r"(under|below|less than|shorter than|over|above|more than|longer than)\s*(\d+(?:\.\d+)?)\s*(?:minutes?|mins?)",
        prompt_lower
    )

    if match:

        operator = match.group(1)
        duration = float(match.group(2))

        if operator in ["under", "below", "less than", "shorter than"]:
            return None, duration

        return duration, None

    return None, None


# Test examples
test_prompts = [
    "Find 10 tracks that are under 2.5 minutes long.",
    "Recommend 5 songs longer than 4 minutes.",
    "Find 3 tracks less than 3 minutes.",
    "Recommend 5 country songs."
]

for prompt in test_prompts:
    operator, duration = get_requested_duration(prompt)

    print(prompt)
    print("Duration requirement:", operator, duration)
    print()

Find 10 tracks that are under 2.5 minutes long.
Duration requirement: None 2.5

Recommend 5 songs longer than 4 minutes.
Duration requirement: 4.0 None

Find 3 tracks less than 3 minutes.
Duration requirement: None 3.0

Recommend 5 country songs.
Duration requirement: None None



In [58]:
def extract_constraints(prompt):

    movie_decade, music_decade = get_requested_decades(prompt)

    movie_genres, music_genres = get_requested_genres(prompt)

    return {
        "movie_decade": movie_decade,
        "music_decade": music_decade,
        "movie_genres": movie_genres,
        "music_genres": music_genres,
        "rating": get_requested_rating(prompt),
        "popularity": get_requested_popularity(prompt),
        "explicit": get_requested_explicit(prompt),
        "duration": get_requested_duration(prompt)
    }


# Test the combined parser
test_prompt = (
    "Recommend 3 Sci-Fi movies from the 1990s rated at least 7.0 "
    "and 5 clean nu metal tracks from the 2000s with popularity 70+ "
    "under 4 minutes."
)

constraints = extract_constraints(test_prompt)

for name, value in constraints.items():
    print(f"{name}: {value}")

movie_decade: (1990, 1999)
music_decade: (2000, 2009)
movie_genres: ['sci-fi']
music_genres: ['nu metal']
rating: 7.0
popularity: 70
explicit: False
duration: ('under', 4.0)


In [27]:
import inspect
from trl import GRPOTrainer, GRPOConfig

print("GRPOTrainer:")
print(inspect.signature(GRPOTrainer.__init__))

print("\nGRPOConfig:")
print(inspect.signature(GRPOConfig.__init__))

GRPOTrainer:
(self, model: Union[str, transformers.modeling_utils.PreTrainedModel], reward_funcs: Union[str, transformers.modeling_utils.PreTrainedModel, Callable[[list, list], list[float]], list[Union[str, transformers.modeling_utils.PreTrainedModel, Callable[[list, list], list[float]]]]], args: Optional[trl.trainer.grpo_config.GRPOConfig] = None, train_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, NoneType] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, dict[str, Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset]], NoneType] = None, processing_class: Optional[transformers.tokenization_utils_base.PreTrainedTokenizerBase] = None, reward_processing_classes: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, list[transformers.tokenization_utils_base.PreTrainedTokenizerBase], NoneType] = None, callbacks: Optional[list[transformers.trainer_callb

In [28]:
import inspect
from trl import GRPOTrainer

# Find the installed GRPOTrainer source file
source_file = inspect.getfile(GRPOTrainer)

print("GRPOTrainer source:", source_file)

# Display relevant reward-function implementation lines
with open(source_file, "r", encoding="utf-8") as file:
    lines = file.readlines()

for i, line in enumerate(lines):
    if "reward_func(" in line or "reward_func =" in line:
        start = max(0, i - 5)
        end = min(len(lines), i + 12)

        print(f"\n--- Source lines {start + 1}-{end} ---")
        print("".join(lines[start:end]))

GRPOTrainer source: /usr/local/lib/python3.13/dist-packages/trl/trainer/grpo_trainer.py

--- Source lines 302-318 ---
    from trl import GRPOTrainer

    dataset = load_dataset("trl-lib/tldr", split="train")


    def reward_func(completions, **kwargs):
        # Dummy reward function that rewards completions with more unique letters.
        return [float(len(set(completion))) for completion in completions]


    trainer = GRPOTrainer(
        model="Qwen/Qwen2-0.5B-Instruct",
        reward_funcs=reward_func,
        train_dataset=dataset,
    )

    trainer.train()


--- Source lines 1020-1036 ---
                    reward_inputs = reward_processing_class(
                        text=texts, return_tensors="pt", padding=True, padding_side="right", add_special_tokens=False
                    )
                    reward_inputs = super()._prepare_inputs(reward_inputs)
                    with torch.inference_mode():
                        rewards_per_func[:, i] = reward_func(**rew

In [29]:
from datasets import Dataset

# Extract user prompts from the training examples
grpo_data = []

for example in examples:

    prompt = example["messages"][-2]["content"]

    grpo_data.append({
        "prompt": prompt
    })

# Convert to Hugging Face Dataset
grpo_dataset = Dataset.from_list(grpo_data)

print("GRPO training examples:", len(grpo_dataset))
print("\nDataset columns:", grpo_dataset.column_names)

print("\nExample prompt:")
print(grpo_dataset[0]["prompt"])

GRPO training examples: 2320

Dataset columns: ['prompt']

Example prompt:
Recommend 3 Sci-Fi movies and 3 nu metal tracks.


In [30]:
# Prepare prompts using Qwen's chat template

grpo_data = []

for example in examples:

    user_prompt = example["messages"][-2]["content"]

    formatted_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    grpo_data.append({
        "prompt": formatted_prompt,
        "user_prompt": user_prompt
    })

grpo_dataset = Dataset.from_list(grpo_data)

print("GRPO training examples:", len(grpo_dataset))
print("Dataset columns:", grpo_dataset.column_names)

print("\nOriginal prompt:")
print(grpo_dataset[0]["user_prompt"])

print("\nFormatted prompt:")
print(grpo_dataset[0]["prompt"])

GRPO training examples: 2320
Dataset columns: ['prompt', 'user_prompt']

Original prompt:
Recommend 3 Sci-Fi movies and 3 nu metal tracks.

Formatted prompt:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Recommend 3 Sci-Fi movies and 3 nu metal tracks.<|im_end|>
<|im_start|>assistant



In [31]:
def extract_recommendation_ids(completion):

    movie_ids = extract_movie_ids(completion)
    spotify_ids = extract_spotify_ids(completion)

    return movie_ids, spotify_ids


# Test using an existing SFT answer
test_answer = examples[0]["messages"][-1]["content"]

movie_ids, spotify_ids = extract_recommendation_ids(test_answer)

print("Movie IDs:", movie_ids)
print("Spotify IDs:", spotify_ids)

print("\nValid movie IDs:",
      sum(1 for movie_id in movie_ids if movie_id in movie_lookup))

print("Valid Spotify IDs:",
      sum(1 for track_id in spotify_ids if track_id in spotify_lookup))

Movie IDs: ['tt0384116', 'tt1677720', 'tt4806232']
Spotify IDs: ['7nAfXgeHfDO50upcOjJOaq', '1mI12kERORxUPtlrQe0OMb', '7AB0cUXnzuSlAnyHOqmrZr']

Valid movie IDs: 3
Valid Spotify IDs: 3


In [149]:
def validity_quantity_reward(prompts, completions, **kwargs):

    rewards = []

    user_prompts = kwargs.get("user_prompt", prompts)

    for index, completion in enumerate(completions):

        user_prompt = user_prompts[index]

        movie_ids, spotify_ids = extract_recommendation_ids(completion)

        movie_count, music_count = get_requested_counts(user_prompt)

        # Count all numbered recommendation lines
        numbered_lines = re.findall(
            r"(?m)^\s*\d+[.)]\s+",
            completion
        )

        total_lines = len(numbered_lines)

        category_rewards = []

        # Movie quantity and validity
        if movie_count is not None:

            unique_movies = list(dict.fromkeys(movie_ids))

            valid_movies = [
                mid for mid in unique_movies
                if mid in movie_lookup
            ]

            movie_reward = min(
                len(valid_movies), movie_count
            ) / movie_count

            category_rewards.append(movie_reward)

        # Music quantity and validity
        if music_count is not None:

            unique_tracks = list(dict.fromkeys(spotify_ids))

            valid_tracks = [
                tid for tid in unique_tracks
                if tid in spotify_lookup
            ]

            music_reward = min(
                len(valid_tracks), music_count
            ) / music_count

            category_rewards.append(music_reward)

        # Average requested category rewards
        reward = 0.0

        if category_rewards:
            reward = sum(category_rewards) / len(category_rewards)

        # Penalize excessive recommendations
        requested_total = (movie_count or 0) + (music_count or 0)

        if requested_total > 0 and total_lines > requested_total:
            excess = total_lines - requested_total
            reward -= 0.2 * (excess / requested_total)

        rewards.append(max(0.0, reward))

    return rewards

In [172]:

def genre_reward(prompts, completions, **kwargs):

    rewards = []

    user_prompts = kwargs.get("user_prompt", prompts)

    if not isinstance(user_prompts, list):
        user_prompts = prompts

    for index, completion in enumerate(completions):

        user_prompt = user_prompts[index]

        requested_movies, requested_music = get_requested_genres(user_prompt)

        movie_count, music_count = get_requested_counts(user_prompt)

        movie_ids, spotify_ids = extract_recommendation_ids(completion)

        # Remove duplicate IDs
        movie_ids = list(dict.fromkeys(movie_ids))
        spotify_ids = list(dict.fromkeys(spotify_ids))

        category_rewards = []

        # Check movie genres
        if requested_movies:

            matching = 0

            for movie_id in movie_ids:

                if movie_id not in movie_lookup:
                    continue

                genres = set(
                    str(movie_lookup[movie_id]["genres"]).lower().split(",")
                )

                genres = {genre.strip() for genre in genres}

                if any(genre in genres for genre in requested_movies):
                    matching += 1

            denominator = movie_count if movie_count else len(movie_ids)

            if denominator > 0:
                category_rewards.append(
                    min(matching, denominator) / denominator
                )
            else:
                category_rewards.append(0.0)

        # Check music genres
        if requested_music:

            matching = 0

            for track_id in spotify_ids:

                if track_id not in spotify_lookup:
                    continue

                genres = set(
                    str(spotify_lookup[track_id]["genres"]).lower().split(",")
                )

                genres = {genre.strip() for genre in genres}

                if any(genre in genres for genre in requested_music):
                    matching += 1

            denominator = music_count if music_count else len(spotify_ids)

            if denominator > 0:
                category_rewards.append(
                    min(matching, denominator) / denominator
                )
            else:
                category_rewards.append(0.0)

        # No genre constraints means this reward is not applicable
        if not category_rewards:
            rewards.append(0.0)
        else:
            rewards.append(sum(category_rewards) / len(category_rewards))

    return rewards


In [187]:
def metadata_reward(prompts, completions, **kwargs):

    rewards = []

    user_prompts = kwargs.get("user_prompt", prompts)

    if not isinstance(user_prompts, list):
        user_prompts = prompts

    for i, completion in enumerate(completions):

        prompt = user_prompts[i]
        constraints = extract_constraints(prompt)

        movie_ids, spotify_ids = extract_recommendation_ids(completion)

        # Remove duplicate IDs
        movie_ids = list(dict.fromkeys(movie_ids))
        spotify_ids = list(dict.fromkeys(spotify_ids))

        movie_count, music_count = get_requested_counts(prompt)

        checks = []

        # Determine how many recommendations should be evaluated
        expected_movies = movie_count if movie_count is not None else len(movie_ids)
        expected_music = music_count if music_count is not None else len(spotify_ids)

        # Helper to add checks and count missing recommendations as failures
        def add_checks(ids, lookup, expected_count, check_function):

            results = []

            for item_id in ids:

                item = lookup.get(item_id)

                if item:
                    results.append(check_function(item))
                else:
                    results.append(False)

            # Missing or unresolved recommendations receive no credit
            missing = max(0, expected_count - len(ids))

            results.extend([False] * missing)

            checks.extend(results)

        # Check movie decades
        movie_decade = constraints["movie_decade"]

        if movie_decade:

            add_checks(
                movie_ids,
                movie_lookup,
                expected_movies,
                lambda movie: (
                    movie_decade[0] <= movie["year"] <= movie_decade[1]
                )
            )

        # Check music decades
        music_decade = constraints["music_decade"]

        if music_decade:

            add_checks(
                spotify_ids,
                spotify_lookup,
                expected_music,
                lambda track: (
                    music_decade[0] <= track["year"] <= music_decade[1]
                )
            )

        # Check movie ratings
        minimum_rating = constraints["rating"]

        if minimum_rating is not None:

            add_checks(
                movie_ids,
                movie_lookup,
                expected_movies,
                lambda movie: movie["rating"] >= minimum_rating
            )

        # Check music popularity
        minimum_popularity = constraints["popularity"]

        if minimum_popularity is not None:

            add_checks(
                spotify_ids,
                spotify_lookup,
                expected_music,
                lambda track: track["popularity"] >= minimum_popularity
            )

        # Check explicit / clean music
        explicit = constraints["explicit"]

        if explicit is not None:

            add_checks(
                spotify_ids,
                spotify_lookup,
                expected_music,
                lambda track: track["explicit"] == explicit
            )

        # Check music duration
        min_duration, max_duration = constraints["duration"]

        if min_duration is not None:
            min_duration = float(min_duration)

        if max_duration is not None:
            max_duration = float(max_duration)

        if min_duration is not None or max_duration is not None:

            def check_duration(track):

                duration = float(track["duration"])

                valid = True

                if min_duration is not None:
                    valid = valid and duration >= float(min_duration)

                if max_duration is not None:
                    valid = valid and duration <= float(max_duration)

                return valid

            add_checks(
                spotify_ids,
                spotify_lookup,
                expected_music,
                check_duration
            )

        # Calculate final metadata reward
        if checks:
            rewards.append(sum(checks) / len(checks))
        else:
            rewards.append(0.0)

    return rewards


In [36]:
# Find suitable examples from the training data

movie_test = None
music_test = None

for example in examples:

    prompt = example["messages"][-2]["content"]
    answer = example["messages"][-1]["content"]

    if movie_test is None:
        if "movie" in prompt.lower() and "1990s" in prompt.lower() and "rated" in prompt.lower():
            movie_test = (prompt, answer)

    if music_test is None:
        if "clean" in prompt.lower() and "popularity" in prompt.lower():
            music_test = (prompt, answer)

    if movie_test is not None and music_test is not None:
        break


# Test movie metadata
if movie_test is not None:

    reward = metadata_reward(
        prompts=[movie_test[0]],
        completions=[movie_test[1]]
    )

    print("MOVIE TEST")
    print("Prompt:", movie_test[0])
    print("Metadata reward:", reward[0])

else:
    print("No matching movie example found.")


# Test music metadata
if music_test is not None:

    reward = metadata_reward(
        prompts=[music_test[0]],
        completions=[music_test[1]]
    )

    print("\nMUSIC TEST")
    print("Prompt:", music_test[0])
    print("Metadata reward:", reward[0])

else:
    print("\nNo matching music example found.")

MOVIE TEST
Prompt: Recommend 10 animation movies from the 1990s rated at least 8.0.
Metadata reward: 0.0

No matching music example found.


In [37]:
# Inspect the movie example
prompt, answer = movie_test

movie_ids, spotify_ids = extract_recommendation_ids(answer)

print("PROMPT:")
print(prompt)

print("\nEXTRACTED MOVIE IDS:")
print(movie_ids)

print("\nDATABASE CHECKS:")

for movie_id in movie_ids:

    if movie_id not in movie_lookup:
        print(movie_id, "- NOT FOUND")
        continue

    movie = movie_lookup[movie_id]

    print(
        movie_id,
        "|", movie["title"],
        "| Year:", movie["year"],
        "| Rating:", movie["rating"],
        "| Decade OK:",
        1990 <= movie["year"] <= 1999,
        "| Rating OK:",
        movie["rating"] >= 8.0
    )

print("\nORIGINAL ANSWER:")
print(answer)


# Find separate music examples
clean_example = None
popularity_example = None

for example in examples:

    prompt = example["messages"][-2]["content"]
    answer = example["messages"][-1]["content"]

    if clean_example is None and "clean" in prompt.lower():
        clean_example = (prompt, answer)

    if popularity_example is None and "popularity" in prompt.lower():
        popularity_example = (prompt, answer)

print("\nCLEAN MUSIC EXAMPLE:")
if clean_example:
    print(clean_example[0])
else:
    print("None found")

print("\nPOPULARITY MUSIC EXAMPLE:")
if popularity_example:
    print(popularity_example[0])
else:
    print("None found")

PROMPT:
Recommend 10 animation movies from the 1990s rated at least 8.0.

EXTRACTED MOVIE IDS:
[]

DATABASE CHECKS:

ORIGINAL ANSWER:
1. Todd McFarlane's Spawn 2 (1998) - Action,Adventure,Animation - IMDb: 8.2
2. Toy Story (1995) - Adventure,Animation,Comedy - IMDb: 8.3
3. Neon Genesis Evangelion: The End of Evangelion (1997) - Action,Animation,Drama - IMDb: 8.1
4. Beauty and the Beast (1991) - Animation,Family,Fantasy - IMDb: 8.0
5. Todd McFarlane's Spawn 3: The Ultimate Battle (1999) - Action,Animation,Drama - IMDb: 8.2
6. The Lion King (1994) - Adventure,Animation,Drama - IMDb: 8.5
7. Aladdin (1992) - Adventure,Animation,Comedy - IMDb: 8.0
8. The Iron Giant (1999) - Action,Adventure,Animation - IMDb: 8.1
9. Perfect Blue (1997) - Animation,Crime,Drama - IMDb: 8.0
10. Ramayana: The Legend of Prince Rama (1993) - Action,Adventure,Animation - IMDb: 9.1

CLEAN MUSIC EXAMPLE:
Find 10 clean r&b pop songs.

POPULARITY MUSIC EXAMPLE:
Recommend 10 tracks by artists with popularity of at least

In [38]:
# Count answers containing different ID formats

movie_id_count = 0
spotify_id_count = 0
total_answers = len(examples)

for example in examples:

    answer = example["messages"][-1]["content"]

    if extract_movie_ids(answer):
        movie_id_count += 1

    if extract_spotify_ids(answer):
        spotify_id_count += 1

print("Total answers:", total_answers)

print(
    "Answers containing IMDb IDs:",
    movie_id_count,
    "/", total_answers
)

print(
    "Answers containing Spotify IDs:",
    spotify_id_count,
    "/", total_answers
)

# Find an example of each answer type
for example in examples:

    answer = example["messages"][-1]["content"]

    if "IMDb:" in answer and not extract_movie_ids(answer):
        print("\nMovie answer without IMDb IDs:")
        print(answer[:1000])
        break

Total answers: 2320
Answers containing IMDb IDs: 320 / 2320
Answers containing Spotify IDs: 1120 / 2320

Movie answer without IMDb IDs:
1. A Sunday in the Country (1984) - Drama - IMDb: 7.4
2. Naram Garam (1981) - Comedy,Drama - IMDb: 7.3
3. The Stubborn Mule (1981) - Comedy,Drama,History - IMDb: 8.1
4. Billionaire (1987) - Comedy,Drama - IMDb: 7.5
5. Monty Python's the Meaning of Life (1983) - Comedy,Musical - IMDb: 7.5
6. The Killing of America (1981) - Crime,Documentary - IMDb: 7.5
7. Mur murs (1981) - Documentary - IMDb: 7.4
8. Visitor of a Museum (1989) - Drama,Horror,Sci-Fi - IMDb: 7.2
9. Kireedam (1989) - Action,Drama - IMDb: 8.8
10. Tight Skin 3 (1988) - Comedy - IMDb: 7.7


In [39]:
import re

def normalize_title(title):
    title = title.lower()
    title = re.sub(r"[^a-z0-9]", "", title)
    return title


# Create title + year lookup
movie_title_lookup = {}

for row in movies_df.itertuples(index=False):

    if pd.isna(row.title) or pd.isna(row.year):
        continue

    key = (normalize_title(row.title), int(row.year))

    if key not in movie_title_lookup:
        movie_title_lookup[key] = []

    movie_title_lookup[key].append(str(row.id))


print("Movie title lookup entries:", len(movie_title_lookup))

Movie title lookup entries: 338021


In [40]:
def extract_movie_ids(completion):

    # First, collect explicitly provided IMDb IDs
    movie_ids = re.findall(
        r"IMDb ID:\s*(tt\d+)",
        completion
    )

    # Then, look for movie titles and release years
    for line in completion.splitlines():

        match = re.search(
            r"^\s*(?:[-*]\s*)?\d+[.)]\s*(.*?)\s*\((\d{4})\)",
            line
        )

        if not match:
            continue

        title = match.group(1).strip()
        year = int(match.group(2))

        key = (normalize_title(title), year)

        if key in movie_title_lookup:
            movie_ids.extend(movie_title_lookup[key])

    # Remove duplicate IDs while preserving order
    return list(dict.fromkeys(movie_ids))


# Test the previously failing movie answer
test_answer = movie_test[1]

movie_ids = extract_movie_ids(test_answer)

print("Extracted movie IDs:", len(movie_ids))

print("\nResolved movies:")

for movie_id in movie_ids:
    print(movie_id, "-", movie_lookup[movie_id]["title"])

Extracted movie IDs: 10

Resolved movies:
tt4641828 - Todd McFarlane's Spawn 2
tt0114709 - Toy Story
tt0169858 - Neon Genesis Evangelion: The End of Evangelion
tt0101414 - Beauty and the Beast
tt4641882 - Todd McFarlane's Spawn 3: The Ultimate Battle
tt0110357 - The Lion King
tt0103639 - Aladdin
tt0129167 - The Iron Giant
tt0156887 - Perfect Blue
tt0259534 - Ramayana: The Legend of Prince Rama


In [47]:
# Normalize titles so punctuation and capitalization differences
# don't prevent a match.
def normalize_title(title):
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        str(title).lower()
    ).strip()


# Build title + year lookup
movie_title_lookup = {}

for movie_id, movie in movie_lookup.items():

    if pd.isna(movie["year"]):
        continue

    title = normalize_title(movie["title"])
    year = int(movie["year"])

    key = (title, year)

    if key not in movie_title_lookup:
        movie_title_lookup[key] = movie_id


# Extract IMDb IDs, including answers without explicit IDs
def extract_movie_ids(completion):

    # First, look for explicitly provided IMDb IDs
    ids = re.findall(r"IMDb ID:\s*(tt\d+)", completion)

    if ids:
        return ids

    # Otherwise, resolve movie titles and years
    resolved_ids = []

    for line in completion.splitlines():

        match = re.search(
            r"^\s*\d+\.\s+(.+?)\s+\((\d{4})\)\s+-",
            line
        )

        if match:

            title = normalize_title(match.group(1))
            year = int(match.group(2))

            movie_id = movie_title_lookup.get((title, year))

            if movie_id:
                resolved_ids.append(movie_id)

    return resolved_ids

In [48]:
movie_ids = extract_movie_ids(answer)

print("Extracted movie IDs:", len(movie_ids))

for movie_id in movie_ids:
    print(movie_id, "-", movie_lookup[movie_id]["title"])

Extracted movie IDs: 10
tt0088318 - A Sunday in the Country
tt0082796 - Naram Garam
tt0082087 - The Stubborn Mule
tt0263743 - Billionaire
tt0085959 - Monty Python's the Meaning of Life
tt0157894 - The Killing of America
tt0081204 - Mur murs
tt0173024 - Visitor of a Museum
tt0237376 - Kireedam
tt0181050 - Tight Skin 3


In [49]:
prompt, answer = movie_test

print("Prompt:", prompt)

print("\nExtracted IDs:")
movie_ids, spotify_ids = extract_recommendation_ids(answer)
print(movie_ids)

print("\nMetadata reward:")
print(metadata_reward([prompt], [answer]))

Prompt: Recommend 10 animation movies from the 1990s rated at least 8.0.

Extracted IDs:
['tt4641828', 'tt0114709', 'tt0169858', 'tt0101414', 'tt4641882', 'tt0110357', 'tt0103639', 'tt0129167', 'tt0156887', 'tt0259534']

Metadata reward:
[1.0]


In [54]:
# Build a Spotify title + artist lookup

spotify_title_artist_lookup = {}

for track_id, track in spotify_lookup.items():

    title = normalize_title(track["title"])
    artist = normalize_title(track["artist"])

    key = (title, artist)

    if key not in spotify_title_artist_lookup:
        spotify_title_artist_lookup[key] = []

    spotify_title_artist_lookup[key].append(track_id)


# Only keep unique title + artist combinations
spotify_title_artist_lookup = {
    key: ids[0]
    for key, ids in spotify_title_artist_lookup.items()
    if len(ids) == 1
}


# Extract movie IDs, including title-based fallback
def extract_movie_ids(completion):

    movie_ids = []

    for line in completion.splitlines():

        # First, look for an explicit IMDb ID
        explicit_match = re.search(
            r"IMDb ID:\s*(tt\d+)",
            line,
            re.IGNORECASE
        )

        if explicit_match:
            movie_ids.append(explicit_match.group(1))
            continue

        # Otherwise, try resolving title + year
        match = re.search(
            r"^\s*(?:[-*]\s*)?\d+[.)]\s*(.*?)\s*\((\d{4})\)",
            line
        )

        if match:

            title = normalize_title(match.group(1))
            year = int(match.group(2))

            movie_id = movie_title_lookup.get((title, year))

            if movie_id:
                movie_ids.append(movie_id)

    return list(dict.fromkeys(movie_ids))


# Extract Spotify IDs, including title + artist fallback
def extract_spotify_ids(completion):

    spotify_ids = []

    for line in completion.splitlines():

        # First, look for an explicit Spotify ID
        explicit_match = re.search(
            r"Spotify ID:\s*([A-Za-z0-9]+)",
            line,
            re.IGNORECASE
        )

        if explicit_match:
            spotify_ids.append(explicit_match.group(1))
            continue

        # Otherwise, extract the title and artist
        entry = line.split("|", 1)[0].strip()

        entry = re.sub(
            r"^\s*(?:[-*]\s*)?\d+[.)]\s*",
            "",
            entry
        )

        if " - " not in entry:
            continue

        title, artist = entry.rsplit(" - ", 1)

        key = (
            normalize_title(title),
            normalize_title(artist)
        )

        track_id = spotify_title_artist_lookup.get(key)

        if track_id:
            spotify_ids.append(track_id)

    return list(dict.fromkeys(spotify_ids))


print("Movie title lookup:", len(movie_title_lookup))
print("Spotify title/artist lookup:", len(spotify_title_artist_lookup))
print("Updated ID extraction functions ready!")

Movie title lookup: 338122
Spotify title/artist lookup: 7304
Updated ID extraction functions ready!


In [55]:
# Check extraction coverage across the entire training dataset

movie_answers = 0
music_answers = 0
movie_resolved = 0
music_resolved = 0

for example in examples:

    prompt = example["messages"][-2]["content"]
    answer = example["messages"][-1]["content"]

    movie_ids, spotify_ids = extract_recommendation_ids(answer)

    if "movie" in prompt.lower() or "film" in prompt.lower():

        movie_answers += 1

        if movie_ids:
            movie_resolved += 1

    if any(word in prompt.lower() for word in ["track", "song", "music"]):

        music_answers += 1

        if spotify_ids:
            music_resolved += 1


print("MOVIE EXTRACTION")
print("Answers:", movie_answers)
print("Answers with resolved IDs:", movie_resolved)

print("\nSPOTIFY EXTRACTION")
print("Answers:", music_answers)
print("Answers with resolved IDs:", music_resolved)

print("\nEXAMPLE EXTRACTION")

for example in examples:

    prompt = example["messages"][-2]["content"]
    answer = example["messages"][-1]["content"]

    if "song" in prompt.lower() or "track" in prompt.lower():

        movie_ids, spotify_ids = extract_recommendation_ids(answer)

        print("Prompt:", prompt)
        print("Spotify IDs:", spotify_ids)
        break

MOVIE EXTRACTION
Answers: 1520
Answers with resolved IDs: 1520

SPOTIFY EXTRACTION
Answers: 1120
Answers with resolved IDs: 1120

EXAMPLE EXTRACTION
Prompt: Recommend 3 Sci-Fi movies and 3 nu metal tracks.
Spotify IDs: ['7nAfXgeHfDO50upcOjJOaq', '1mI12kERORxUPtlrQe0OMb', '7AB0cUXnzuSlAnyHOqmrZr']


In [56]:
def get_requested_decades(prompt):

    movie_decade = None
    music_decade = None

    parts = re.split(r"\band\b", prompt.lower())

    for part in parts:

        match = re.search(r"\b(19\d0s|20\d0s)\b", part)

        if not match:
            continue

        start_year = int(match.group(1)[:-1])
        end_year = start_year + 9

        has_movie = any(
            word in part for word in ["movie", "film"]
        )

        has_music = any(
            word in part for word in ["track", "song", "music"]
        )

        if has_movie:
            movie_decade = (start_year, end_year)

        if has_music:
            music_decade = (start_year, end_year)

    # If only one decade was requested, apply it to all
    # requested categories when the prompt is unambiguous.
    all_decades = re.findall(
        r"\b(19\d0s|20\d0s)\b",
        prompt.lower()
    )

    if len(all_decades) == 1:

        start_year = int(all_decades[0][:-1])
        decade = (start_year, start_year + 9)

        if movie_decade is None and any(
            word in prompt.lower() for word in ["movie", "film"]
        ):
            movie_decade = decade

        if music_decade is None and any(
            word in prompt.lower() for word in ["track", "song", "music"]
        ):
            music_decade = decade

    return movie_decade, music_decade

In [59]:
tests = [
    "Recommend 3 movies from the 1990s.",
    "Recommend 5 tracks from the 2010s.",
    "Recommend 3 movies from the 1990s and 5 tracks from the 2000s.",
    "Recommend 3 movies and 5 tracks from the 1990s."
]

for prompt in tests:

    movie_decade, music_decade = get_requested_decades(prompt)

    print("\nPrompt:", prompt)
    print("Movie decade:", movie_decade)
    print("Music decade:", music_decade)


Prompt: Recommend 3 movies from the 1990s.
Movie decade: (1990, 1999)
Music decade: None

Prompt: Recommend 5 tracks from the 2010s.
Movie decade: None
Music decade: (2010, 2019)

Prompt: Recommend 3 movies from the 1990s and 5 tracks from the 2000s.
Movie decade: (1990, 1999)
Music decade: (2000, 2009)

Prompt: Recommend 3 movies and 5 tracks from the 1990s.
Movie decade: (1990, 1999)
Music decade: (1990, 1999)


In [60]:
test_prompts = [
    "Recommend 3 movies from the 1990s.",
    "Recommend 5 tracks from the 2010s.",
    "Recommend 3 movies from the 1990s and 5 tracks from the 2000s."
]

for prompt in test_prompts:

    print("\nPrompt:", prompt)
    print(extract_constraints(prompt))


Prompt: Recommend 3 movies from the 1990s.
{'movie_decade': (1990, 1999), 'music_decade': None, 'movie_genres': [], 'music_genres': [], 'rating': None, 'popularity': None, 'explicit': None, 'duration': (None, None)}

Prompt: Recommend 5 tracks from the 2010s.
{'movie_decade': None, 'music_decade': (2010, 2019), 'movie_genres': [], 'music_genres': [], 'rating': None, 'popularity': None, 'explicit': None, 'duration': (None, None)}

Prompt: Recommend 3 movies from the 1990s and 5 tracks from the 2000s.
{'movie_decade': (1990, 1999), 'music_decade': (2000, 2009), 'movie_genres': [], 'music_genres': [], 'rating': None, 'popularity': None, 'explicit': None, 'duration': (None, None)}


In [62]:
prompt = "Recommend 10 animation movies from the 1990s rated at least 8.0."

print("Prompt:", prompt)
print("Constraints:", extract_constraints(prompt))

movie_ids = extract_movie_ids(answer)

print("\nExtracted movie IDs:", len(movie_ids))
print("Metadata reward:", metadata_reward([prompt], [answer]))

Prompt: Recommend 10 animation movies from the 1990s rated at least 8.0.
Constraints: {'movie_decade': (1990, 1999), 'music_decade': None, 'movie_genres': ['animation'], 'music_genres': [], 'rating': 8.0, 'popularity': None, 'explicit': None, 'duration': (None, None)}

Extracted movie IDs: 3
Metadata reward: [0.3333333333333333]


In [63]:
prompt = "Recommend 10 animation movies from the 1990s rated at least 8.0."

constraints = extract_constraints(prompt)
movie_ids, spotify_ids = extract_recommendation_ids(answer)

print("Extracted movies:\n")

for movie_id in movie_ids:

    movie = movie_lookup.get(movie_id)

    if movie is None:
        print(movie_id, "NOT FOUND")
        continue

    decade_ok = (
        constraints["movie_decade"][0]
        <= movie["year"]
        <= constraints["movie_decade"][1]
    )

    rating_ok = movie["rating"] >= constraints["rating"]

    print(
        movie["title"],
        "| Year:", movie["year"],
        "| Rating:", movie["rating"],
        "| Decade:", decade_ok,
        "| Rating check:", rating_ok
    )

print("\nFinal metadata reward:")
print(metadata_reward([prompt], [answer]))

Extracted movies:

G.O.R.A. | Year: 2004 | Rating: 8.0 | Decade: False | Rating check: True
Ready Player One | Year: 2018 | Rating: 7.4 | Decade: False | Rating check: False
Indru Netru Naalai | Year: 2015 | Rating: 8.0 | Decade: False | Rating check: True

Final metadata reward:
[0.3333333333333333]


In [64]:
prompt, answer = movie_test

print("PROMPT:")
print(prompt)

print("\nANSWER:")
print(answer)

print("\nEXTRACTED IDS:")
print(extract_movie_ids(answer))

PROMPT:
Recommend 10 animation movies from the 1990s rated at least 8.0.

ANSWER:
1. Todd McFarlane's Spawn 2 (1998) - Action,Adventure,Animation - IMDb: 8.2
2. Toy Story (1995) - Adventure,Animation,Comedy - IMDb: 8.3
3. Neon Genesis Evangelion: The End of Evangelion (1997) - Action,Animation,Drama - IMDb: 8.1
4. Beauty and the Beast (1991) - Animation,Family,Fantasy - IMDb: 8.0
5. Todd McFarlane's Spawn 3: The Ultimate Battle (1999) - Action,Animation,Drama - IMDb: 8.2
6. The Lion King (1994) - Adventure,Animation,Drama - IMDb: 8.5
7. Aladdin (1992) - Adventure,Animation,Comedy - IMDb: 8.0
8. The Iron Giant (1999) - Action,Adventure,Animation - IMDb: 8.1
9. Perfect Blue (1997) - Animation,Crime,Drama - IMDb: 8.0
10. Ramayana: The Legend of Prince Rama (1993) - Action,Adventure,Animation - IMDb: 9.1

EXTRACTED IDS:
['tt4641828', 'tt0114709', 'tt0169858', 'tt0101414', 'tt4641882', 'tt0110357', 'tt0103639', 'tt0129167', 'tt0156887', 'tt0259534']


In [65]:
prompt, answer = movie_test

print("Extracted movie IDs:", len(extract_movie_ids(answer)))

print("\nMetadata reward:")
print(metadata_reward([prompt], [answer]))

print("\nIndividual checks:")

constraints = extract_constraints(prompt)
movie_ids = extract_movie_ids(answer)

for movie_id in movie_ids:

    movie = movie_lookup.get(movie_id)

    if movie is None:
        print(movie_id, "NOT FOUND")
        continue

    decade_ok = (
        constraints["movie_decade"][0]
        <= movie["year"]
        <= constraints["movie_decade"][1]
    )

    rating_ok = movie["rating"] >= constraints["rating"]

    print(
        movie["title"],
        "|", movie["year"],
        "|", movie["rating"],
        "| Decade:", decade_ok,
        "| Rating:", rating_ok
    )

Extracted movie IDs: 10

Metadata reward:
[1.0]

Individual checks:
Todd McFarlane's Spawn 2 | 1998 | 8.2 | Decade: True | Rating: True
Toy Story | 1995 | 8.3 | Decade: True | Rating: True
Neon Genesis Evangelion: The End of Evangelion | 1997 | 8.1 | Decade: True | Rating: True
Beauty and the Beast | 1991 | 8.0 | Decade: True | Rating: True
Todd McFarlane's Spawn 3: The Ultimate Battle | 1999 | 8.2 | Decade: True | Rating: True
The Lion King | 1994 | 8.5 | Decade: True | Rating: True
Aladdin | 1992 | 8.0 | Decade: True | Rating: True
The Iron Giant | 1999 | 8.1 | Decade: True | Rating: True
Perfect Blue | 1997 | 8.0 | Decade: True | Rating: True
Ramayana: The Legend of Prince Rama | 1993 | 9.1 | Decade: True | Rating: True


In [68]:
music_test = None

for example in examples:

    prompt = example["messages"][-2]["content"]
    answer = example["messages"][-1]["content"]

    is_music_prompt = any(
        word in prompt.lower()
        for word in ["track", "song", "music"]
    )

    spotify_ids = extract_spotify_ids(answer)

    if is_music_prompt and len(spotify_ids) > 0:

        music_test = (prompt, answer)
        break


if music_test is None:
    print("No usable music example found.")

else:
    music_prompt, music_answer = music_test

    print("Prompt:", music_prompt)
    print("\nAnswer:")
    print(music_answer)

    print("\nExtracted Spotify IDs:")
    print(extract_spotify_ids(music_answer))

Prompt: Recommend 3 Sci-Fi movies and 3 nu metal tracks.

Answer:
Movies:
1. G.O.R.A. (2004) | Genres: Adventure,Comedy,Sci-Fi | Rating: 8.0 | Votes: 72897 | IMDb ID: tt0384116
2. Ready Player One (2018) | Genres: Action,Adventure,Sci-Fi | Rating: 7.4 | Votes: 555427 | IMDb ID: tt1677720
3. Indru Netru Naalai (2015) | Genres: Comedy,Sci-Fi,Thriller | Rating: 8.0 | Votes: 5702 | IMDb ID: tt4806232

Music:
1. Talking to Myself - Linkin Park | Genres: nu metal, rap metal, rock, alternative metal | Popularity: 60 | Year: 2017 | Duration: 3.85 min | Explicit: No | Spotify ID: 7nAfXgeHfDO50upcOjJOaq
2. Posthuman - Marilyn Manson | Genres: industrial metal, industrial rock, industrial, metal, nu metal | Popularity: 40 | Year: 1998 | Duration: 4.29 min | Explicit: Yes | Spotify ID: 1mI12kERORxUPtlrQe0OMb
3. Faint - Linkin Park | Genres: nu metal, rap metal, rock, alternative metal | Popularity: 70 | Year: 2003 | Duration: 2.70 min | Explicit: No | Spotify ID: 7AB0cUXnzuSlAnyHOqmrZr

Extracted 

In [71]:
music_prompt, music_answer = music_test

print("MUSIC TEST")
print("Prompt:", music_prompt)

print("\nExtracted Spotify IDs:")
print(extract_spotify_ids(music_answer))

print("\nQuantity reward:")
print(validity_quantity_reward([music_prompt], [music_answer]))

print("\nGenre reward:")
print(genre_reward([music_prompt], [music_answer]))

print("\nMetadata reward:")
print(metadata_reward([music_prompt], [music_answer]))

MUSIC TEST
Prompt: Recommend 3 Sci-Fi movies and 3 nu metal tracks.

Extracted Spotify IDs:
['7nAfXgeHfDO50upcOjJOaq', '1mI12kERORxUPtlrQe0OMb', '7AB0cUXnzuSlAnyHOqmrZr']

Quantity reward:
[1.0]

Genre reward:
[1.0]

Metadata reward:
[1.0]


In [72]:
print("Quantity reward:", validity_quantity_reward)
print("Genre reward:", genre_reward)
print("Metadata reward:", metadata_reward)

print("\nGRPO dataset size:", len(grpo_dataset))
print("Training examples:", len(examples))

print("\nExample GRPO dataset entry:")
print(grpo_dataset[0])

Quantity reward: <function validity_quantity_reward at 0x7c0d0c8a36a0>
Genre reward: <function genre_reward at 0x7c0d0c8a25c0>
Metadata reward: <function metadata_reward at 0x7c0d0a273740>

GRPO dataset size: 2320
Training examples: 2320

Example GRPO dataset entry:
{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nRecommend 3 Sci-Fi movies and 3 nu metal tracks.<|im_end|>\n<|im_start|>assistant\n', 'user_prompt': 'Recommend 3 Sci-Fi movies and 3 nu metal tracks.'}


In [73]:
import inspect
from trl import GRPOTrainer

source_file = inspect.getfile(GRPOTrainer)

with open(source_file, "r", encoding="utf-8") as file:
    source = file.read()

search_terms = [
    "reward_kwargs",
    "column_names",
    "prompts=prompts",
    "completions=completions"
]

for term in search_terms:

    print("\nSEARCH:", term)

    position = source.find(term)

    if position == -1:
        print("Not found")
    else:
        print(source[max(0, position - 500):position + 1000])


SEARCH: reward_kwargs
_batch)
        return inputs

    @profiling_decorator
    def _calculate_rewards(self, inputs, prompts, completions, completion_ids_list):
        device = self.accelerator.device
        rewards_per_func = torch.zeros(len(prompts), len(self.reward_funcs), device=device)

        # Repeat all input columns (but "prompt", "completion", and "completion_ids") to match the num of generations
        keys = [key for key in inputs[0] if key not in ["prompt", "completion", "completion_ids"]]
        reward_kwargs = {key: [example[key] for example in inputs] for key in keys}

        for i, (reward_func, reward_processing_class, reward_func_name) in enumerate(
            zip(self.reward_funcs, self.reward_processing_classes, self.reward_func_names)
        ):
            with profiling_context(self, reward_func_name):
                if isinstance(reward_func, nn.Module):  # Module (no PretrainedModel) for compat with compiled models
                    if is_conversa

In [118]:

from trl import GRPOConfig

grpo_config = GRPOConfig(

    output_dir="/content/drive/MyDrive/ReelRhythm/grpo_checkpoints",

    # Training
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # Generation
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=256,
    temperature=0.9,

    # GRPO
    beta=0.04,
    reward_weights=[1.0, 1.0, 1.0, 0.5],

    # Memory
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    # Logging and saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=10,
    logging_steps=5,

    # Other
    optim="adamw_torch",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    seed=42
)

print("GRPO configuration created!")
print("Generations:", grpo_config.num_generations)
print("Reward weights:", grpo_config.reward_weights)
print("KL penalty:", grpo_config.beta)
print("Batch size:", grpo_config.per_device_train_batch_size)


GRPO configuration created!
Generations: 4
Reward weights: [1.0, 1.0, 1.0, 0.5]
KL penalty: 0.04
Batch size: 1


In [119]:
from trl import GRPOTrainer

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[
        validity_quantity_reward,
        genre_reward,
        metadata_reward,
        format_reward
    ],
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer
)

print("GRPO trainer created successfully!")

print("\nReward functions:")
for reward_func in grpo_trainer.reward_funcs:
    print("-", reward_func.__name__)

print("\nTraining examples:", len(grpo_trainer.train_dataset))
print("Reward weights:", grpo_config.reward_weights)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


GRPO trainer created successfully!

Reward functions:
- validity_quantity_reward
- genre_reward
- metadata_reward
- format_reward

Training examples: 2320
Reward weights: [1.0, 1.0, 1.0, 0.5]


In [76]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Total VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

print("Allocated VRAM:",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2),
      "GB")

print("Reserved VRAM:",
      round(torch.cuda.memory_reserved(0) / 1024**3, 2),
      "GB")

GPU: Tesla T4
Total VRAM: 14.56 GB
Allocated VRAM: 0.96 GB
Reserved VRAM: 1.25 GB


In [77]:
print("Model type:", type(grpo_trainer.model).__name__)

print("Reference model exists:",
      grpo_trainer.ref_model is not None)

print("Reference model type:",
      type(grpo_trainer.ref_model).__name__
      if grpo_trainer.ref_model is not None
      else None)

print("KL penalty:", grpo_trainer.args.beta)

Model type: PeftModelForCausalLM
Reference model exists: False
Reference model type: None
KL penalty: 0.04


In [78]:
from peft import LoraConfig, get_peft_model

# Merge the existing SFT adapter into the base model
model = model.merge_and_unload()

print("SFT adapter merged into base model.")

# Create a fresh adapter for GRPO training
grpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, grpo_lora_config)

model.print_trainable_parameters()

SFT adapter merged into base model.


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [79]:
print("Model type:", type(model).__name__)

print("Has PEFT config:", hasattr(model, "peft_config"))

if hasattr(model, "peft_config"):
    print("Adapter names:", list(model.peft_config.keys()))

if hasattr(model, "active_adapter"):
    print("Active adapter:", model.active_adapter)

print("Trainable parameters:")
model.print_trainable_parameters()

Model type: PeftModelForCausalLM
Has PEFT config: True
Adapter names: ['default']
Active adapter: default
Trainable parameters:
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [80]:
import gc
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Release the previous trainer and model
del grpo_trainer
del model

gc.collect()
torch.cuda.empty_cache()

# Reload the original base model
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load the saved SFT adapter
sft_model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/ReelRhythm/sft_adapter"
)

print("Before merging:", type(sft_model).__name__)

# Merge SFT adapter into the base model
merged_sft_model = sft_model.merge_and_unload()

print("After merging:", type(merged_sft_model).__name__)
print("Has PEFT config:", hasattr(merged_sft_model, "peft_config"))

Before merging: PeftModelForCausalLM
After merging: Qwen2ForCausalLM
Has PEFT config: True


In [81]:
print("Model type:", type(merged_sft_model).__name__)

print("PEFT config value:",
      getattr(merged_sft_model, "peft_config", "Not present"))

print("Model config:", merged_sft_model.config.model_type)

print("Trainable parameters:",
      sum(p.numel() for p in merged_sft_model.parameters()
          if p.requires_grad))

Model type: Qwen2ForCausalLM
PEFT config value: {'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='Qwen/Qwen2.5-0.5B-Instruct', revision=None, inference_mode=True, r=16, target_modules={'v_proj', 'o_proj', 'k_proj', 'up_proj', 'down_proj', 'q_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)}
Model config: qwen2
Trainable parameters: 0


In [82]:
# Remove leftover PEFT metadata from the merged model
if hasattr(merged_sft_model, "peft_config"):
    delattr(merged_sft_model, "peft_config")

# Freeze the merged SFT model
for param in merged_sft_model.parameters():
    param.requires_grad = False

print("Model type:", type(merged_sft_model).__name__)
print("Has PEFT config:", hasattr(merged_sft_model, "peft_config"))

# Attach a fresh LoRA adapter for GRPO
from peft import get_peft_model

model = get_peft_model(
    merged_sft_model,
    grpo_lora_config
)

print("\nGRPO model:")
print("Model type:", type(model).__name__)
model.print_trainable_parameters()

Model type: Qwen2ForCausalLM
Has PEFT config: False

GRPO model:
Model type: PeftModelForCausalLM
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [97]:
grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[
        validity_quantity_reward,
        genre_reward,
        metadata_reward
    ],
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer
)

print("GRPO trainer created successfully!")
print("Training examples:", len(grpo_dataset))
print("KL penalty:", grpo_trainer.args.beta)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


GRPO trainer created successfully!
Training examples: 2320
KL penalty: 0.04
